# Earthquake Machine Learning Analysis

This notebook focuses on the USGS earthquake component of a wider MSc Data Mining and Machine Learning group project.

Tasks:
1. Data cleaning and exploratory analysis
2. Random Forest regression for earthquake magnitude estimation
3. Evaluation with RMSE, MAE and R²
4. Feature importance
5. K-Means clustering using geospatial/depth features


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
DATA_PATH = PROJECT_ROOT / "data" / "usgs_all_month_2025-11-24.csv"


## Load the USGS earthquake snapshot


In [ ]:
quake_df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="skip")
print("Dataset shape:", quake_df.shape)
quake_df.head()


## Clean and select numeric features


In [ ]:
numeric_cols = ["latitude", "longitude", "depth", "mag", "gap", "dmin", "rms"]
quake_numeric = quake_df[numeric_cols].dropna().copy()
print("Cleaned shape:", quake_numeric.shape)
quake_numeric.describe()


## Random Forest regression


In [ ]:
X = quake_numeric.drop(columns="mag")
y = quake_numeric["mag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")


## Feature importance

Feature importance is a model-specific predictive measure and should not be interpreted as proof of a physical causal relationship.


In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)


## K-Means clustering


In [ ]:
cluster_features = ["latitude", "longitude", "depth"]
cluster_df = quake_numeric[cluster_features].copy()

cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(cluster_df)

cluster_range = range(2, 8)
inertias = []

for k in cluster_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(cluster_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(cluster_range), inertias, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans.fit_predict(cluster_scaled)

plt.figure(figsize=(8, 6))
for cluster_id in sorted(cluster_df["cluster"].unique()):
    subset = cluster_df[cluster_df["cluster"] == cluster_id]
    plt.scatter(subset["longitude"], subset["latitude"], s=10, label=f"Cluster {cluster_id}")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Earthquake Clusters from K-Means")
plt.legend()
plt.show()


## Results

- R²: **0.905**
- RMSE: **0.417**
- MAE: **0.306**
- Highest feature importance in this reproduced model: **dmin**
- K-Means clusters retained: **4**
